# 03 - Improve and explain the fraud model

This notebook builds on `02_baseline_model`. It:

1. Builds **card-history features in SQL** (DuckDB window functions) that only look at *earlier* transactions, so nothing leaks from the future.
2. Compares three models on the same time-based split: logistic regression, the LightGBM baseline, and LightGBM with the new SQL features.
3. Picks a **decision threshold from business costs** instead of the default 0.5.
4. Explains the model with **SHAP**.

Before you run it, shut down any other notebook that has `fraud.duckdb` open (Running tab > Shut Down), otherwise DuckDB will report that the file is in use. This notebook only reads the database.

In [ ]:
import warnings
from pathlib import Path

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (PrecisionRecallDisplay, average_precision_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DB = "../data/fraud.duckdb"
SQL_DIR = Path("../sql")
IMG_DIR = Path("../images")
SQL_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

## 1. Card-history features in SQL

`card1` and `card2` together are only a rough stand-in for "the same card", because the data is anonymised. Every window below looks strictly **backwards** in time (`1 PRECEDING` or earlier), so a transaction never sees itself or the future.

In [ ]:
FEATURE_SQL = """
SELECT
    TransactionID,
    COUNT(*) OVER w24                                          AS txns_prev_24h,
    COALESCE(SUM(TransactionAmt) OVER w24, 0)                  AS amt_prev_24h,
    TransactionDT - LAG(TransactionDT) OVER wseq               AS secs_since_prev_txn,
    TransactionAmt / NULLIF(AVG(TransactionAmt) OVER wpast, 0) AS amt_vs_card_avg,
    (TransactionDT // 3600) % 24                               AS hour_of_day
FROM train
WINDOW
    w24   AS (PARTITION BY card1, card2 ORDER BY TransactionDT
              RANGE BETWEEN 86400 PRECEDING AND 1 PRECEDING),
    wseq  AS (PARTITION BY card1, card2 ORDER BY TransactionDT, TransactionID),
    wpast AS (PARTITION BY card1, card2 ORDER BY TransactionDT, TransactionID
              ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
"""

# keep the query in the repo too, so the SQL is visible on GitHub
(SQL_DIR / "04_card_velocity_features.sql").write_text(FEATURE_SQL.strip() + ";\n")

con = duckdb.connect(DB, read_only=True)
df = con.execute("SELECT * FROM train").df()
feats = con.execute(FEATURE_SQL).df()
con.close()

df = df.merge(feats, on="TransactionID", how="left")
df = df.sort_values("TransactionDT").reset_index(drop=True)
del feats

for c in df.select_dtypes("float64").columns:
    df[c] = df[c].astype("float32")

NEW = ["txns_prev_24h", "amt_prev_24h", "secs_since_prev_txn", "amt_vs_card_avg", "hour_of_day"]
DROP = ["isFraud", "TransactionID", "TransactionDT"]
base_features = [c for c in df.columns if c not in DROP + NEW]
for c in base_features:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("category")
all_features = base_features + NEW

df[NEW + ["isFraud"]].groupby(pd.cut(df["txns_prev_24h"], [-1, 0, 2, 5, np.inf],
                                     labels=["0", "1-2", "3-5", "6+"]), observed=True)["isFraud"].agg(["size", "mean"])

The table above is the same card-velocity idea as `03_card_velocity.sql`, now computed for every row: fraud rate by how many transactions the card made in the previous 24 hours.

## 2. Time-based split (same as the baseline)

Train on the earliest 80% of transactions and test on the latest 20%.

In [ ]:
cut = df["TransactionDT"].quantile(0.8)
train = df[df["TransactionDT"] <= cut]
test = df[df["TransactionDT"] > cut]
y_tr, y_te = train["isFraud"], test["isFraud"]

def scores(name, p):
    return {"model": name,
            "PR-AUC": round(average_precision_score(y_te, p), 4),
            "ROC-AUC": round(roc_auc_score(y_te, p), 4)}

print(f"train rows: {len(train):,}   test rows: {len(test):,}")
print(f"fraud rate in test (= PR-AUC of random guessing): {y_te.mean():.4f}")

## 3. Three models, one split

* **Logistic regression** on the numeric columns only (median-fill, scaled). It is fitted on a 150,000-row sample of the training period to keep memory use and run time reasonable.
* **LightGBM baseline** with the original columns.
* **LightGBM + SQL features** with the five card-history features added.

In [ ]:
results = []

# 1) logistic regression baseline
num_cols = [c for c in base_features if pd.api.types.is_numeric_dtype(df[c])] + NEW
sample = train.sample(n=min(150_000, len(train)), random_state=42)
logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=300))
logit.fit(sample[num_cols], sample["isFraud"])
results.append(scores("Logistic regression (numeric columns)", logit.predict_proba(test[num_cols])[:, 1]))

# 2) and 3) LightGBM without / with the SQL features
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=64, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

m_base = lgb.LGBMClassifier(**PARAMS).fit(train[base_features], y_tr)
results.append(scores("LightGBM baseline", m_base.predict_proba(test[base_features])[:, 1]))

m_new = lgb.LGBMClassifier(**PARAMS).fit(train[all_features], y_tr)
results.append(scores("LightGBM + SQL card features", m_new.predict_proba(test[all_features])[:, 1]))

results_df = pd.DataFrame(results)
results_df

Read the **PR-AUC** column first (random guessing scores the test fraud rate printed above). If the SQL features did not help, that is a legitimate result: say so in the README and keep the simpler model.

## 4. Choosing a threshold from costs

A model gives a fraud *probability*. The business has to decide above which probability to flag a transaction. Two assumed costs:

* flagging a transaction costs a fixed **review cost** (an analyst checks it), and
* missing a fraud costs the **transaction amount**.

`REVIEW_COST` is an assumption, not a number from the data. Change it and re-run to see how the best threshold moves.

To avoid choosing the threshold on the test set, the training period is split again by time: fit on the earliest 87.5% of it (70% of all data), choose the threshold on the next 12.5% (10% of all data), then report on the untouched test set.

In [ ]:
REVIEW_COST = 5.0   # assumed cost of reviewing one flagged transaction (same units as TransactionAmt)

use_new = results[2]["PR-AUC"] >= results[1]["PR-AUC"]
best_cols = all_features if use_new else base_features
print("Using the SQL features:" , use_new)

cut_inner = train["TransactionDT"].quantile(0.875)
fit_df = train[train["TransactionDT"] <= cut_inner]
thr_df = train[train["TransactionDT"] > cut_inner]

model = lgb.LGBMClassifier(**PARAMS).fit(fit_df[best_cols], fit_df["isFraud"])
p_thr = model.predict_proba(thr_df[best_cols])[:, 1]
p_test = model.predict_proba(test[best_cols])[:, 1]

def total_cost(y, amt, p, threshold):
    flagged = p >= threshold
    missed = (~flagged) & (y == 1)
    return REVIEW_COST * flagged.sum() + amt[missed].sum()

y_thr, amt_thr = thr_df["isFraud"].to_numpy(), thr_df["TransactionAmt"].to_numpy()
y_t, amt_t = y_te.to_numpy(), test["TransactionAmt"].to_numpy()

grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
costs = np.array([total_cost(y_thr, amt_thr, p_thr, t) for t in grid])
best_t = float(grid[costs.argmin()])

cost_model = total_cost(y_t, amt_t, p_test, best_t)
cost_nothing = amt_t[y_t == 1].sum()          # flag nothing: every fraud is missed
flagged = p_test >= best_t
saving = 1 - cost_model / cost_nothing

print(f"Best threshold (chosen on validation period): {best_t:.2f}")
print(f"Test set: flags {flagged.mean():.2%} of transactions, "
      f"precision {precision_score(y_t, flagged):.2f}, recall {recall_score(y_t, flagged):.2f}")
print(f"Cost with the model: {cost_model:,.0f}   cost of flagging nothing: {cost_nothing:,.0f}   saving: {saving:.1%}")

plt.figure(figsize=(6, 3.5))
plt.plot(grid, costs)
plt.axvline(best_t, linestyle="--", color="grey")
plt.xlabel("Decision threshold")
plt.ylabel("Total cost on validation period")
plt.title("Cost-based threshold choice")
plt.tight_layout()
plt.savefig(IMG_DIR / "threshold_cost.png", dpi=150)
plt.show()

PrecisionRecallDisplay.from_predictions(y_t, p_test)
plt.title("Precision-recall curve (test period)")
plt.tight_layout()
plt.savefig(IMG_DIR / "pr_curve.png", dpi=150)
plt.show()

## 5. SHAP: what drives the predictions?

SHAP shows how much each feature pushed a prediction up or down. It uses a random sample of up to 5,000 test transactions to keep it fast. Many columns in this dataset are anonymised (V1, V2, C1, D1...), so describe patterns, not meanings you cannot verify.

In [ ]:
import shap

Xs = test[best_cols].sample(n=min(5000, len(test)), random_state=42)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xs)
if isinstance(sv, list):
    sv = sv[1]
sv = np.asarray(sv)
if sv.ndim == 3:
    sv = sv[:, :, 1]

shap.summary_plot(sv, Xs, max_display=15, show=False)
plt.tight_layout()
plt.savefig(IMG_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

top = pd.Series(np.abs(sv).mean(axis=0), index=best_cols).sort_values(ascending=False).head(10)
top.rename("mean |SHAP|")

## 6. Numbers for the README

Run this cell and paste its output into the README, then edit the wording to match what you actually see.

In [ ]:
print("| Model | PR-AUC | ROC-AUC |")
print("| --- | --- | --- |")
for r in results:
    print(f"| {r['model']} | {r['PR-AUC']} | {r['ROC-AUC']} |")
print()
print(f"- Fraud rate in test period: {y_te.mean():.2%} (random-guess PR-AUC {y_te.mean():.3f})")
print(f"- Cost-based threshold {best_t:.2f} (assumed review cost {REVIEW_COST:g} per flag): "
      f"flags {flagged.mean():.1%} of transactions, precision {precision_score(y_t, flagged):.2f}, "
      f"recall {recall_score(y_t, flagged):.2f}, total cost {saving:.1%} lower than flagging nothing")
print("- Top SHAP features:", ", ".join(top.index[:5]))